# AgriDiagnose Model V2 Experiment B — Kaggle workflow

Experiment B changes only the TRAIN augmentation magnitudes. The MobileNetV2 architecture, data manifests, optimization policy, two training phases, and VALIDATION-only candidate selection remain identical to Experiment A. Training is disabled by default. INTERNAL TEST and PlantDoc TEST must not be attached, loaded, or evaluated.

In [ ]:
# 1. Audit the Kaggle system kernel. This is not the Experiment B runtime.
import json, os, platform, re, shutil, subprocess, sys
from pathlib import Path
import tensorflow as system_tf
import keras as system_keras
import numpy as system_numpy

print('System Python:', sys.version)
print('System OS:', platform.platform())
print('System TensorFlow:', system_tf.__version__)
print('System Keras:', system_keras.__version__)
print('System NumPy:', system_numpy.__version__)
print('System CUDA build:', system_tf.test.is_built_with_cuda())
SYSTEM_GPUS = system_tf.config.list_physical_devices('GPU')
print('System GPUs:', SYSTEM_GPUS)
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
if not SYSTEM_GPUS:
    raise RuntimeError('KAGGLE_GPU_NOT_AVAILABLE: Settings -> Accelerator -> GPU')

## 2. Clone the reviewed Experiment B revision

The notebook is pinned to the immutable Experiment B resume implementation commit reviewed below. Never replace it with a moving branch name.

In [ ]:
REPOSITORY_URL = 'https://github.com/ihebjdey2/ai-plant-disease-detection.git'
APPROVED_CODE_REVISION = '1960e63d6eb8049d9b005bbbed377a1db085310d'
PROJECT_ROOT = Path('/kaggle/working/ai-plant-disease-detection')
if not re.fullmatch(r'[0-9a-f]{40}', APPROVED_CODE_REVISION):
    raise RuntimeError('APPROVED_CODE_REVISION_NOT_PINNED')
if not (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'checkout', '--detach', APPROVED_CODE_REVISION], cwd=PROJECT_ROOT, check=True)
HEAD = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.strip()
assert HEAD == APPROVED_CODE_REVISION
print('Approved Experiment B revision:', HEAD)

## 3. Bootstrap the isolated Python 3.11 / TensorFlow 2.15 runtime

The Kaggle Python kernel remains untouched. All Experiment B ML commands run in the same pinned isolated stack used by the Experiment A baseline.

In [ ]:
RUNTIME_ROOT = Path('/kaggle/working/agridiagnose-tf215-runtime')
TF215_PYTHON = RUNTIME_ROOT / 'venvs/agridiagnose-tf215/bin/python'
ISOLATED_ENV = os.environ.copy()
for name in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    ISOLATED_ENV.pop(name, None)
ISOLATED_ENV['PYTHONNOUSERSITE'] = '1'
ISOLATED_ENV['MPLBACKEND'] = 'Agg'
subprocess.run([
    sys.executable, '-I', str(PROJECT_ROOT / 'scripts/bootstrap_kaggle_tf215_runtime.py'),
    '--working-root', str(RUNTIME_ROOT), '--project-root', str(PROJECT_ROOT),
], check=True, env=ISOLATED_ENV)
assert TF215_PYTHON.is_file()
print('Isolated interpreter:', TF215_PYTHON)

## 4. Hard TensorFlow 2.15 GPU gate

The isolated process must validate Python 3.11, TensorFlow/Keras 2.15, NumPy 1.26.4, CUDA, a TensorFlow GPU, and real `/GPU:0` execution.

In [ ]:
RUNTIME_REPORT = RUNTIME_ROOT / 'tf215-gpu-runtime-exp-b.json'
subprocess.run([
    str(TF215_PYTHON), str(PROJECT_ROOT / 'scripts/run_kaggle_model_v2_experiment_b.py'),
    'verify-runtime', '--output', str(RUNTIME_REPORT),
], check=True, env=ISOLATED_ENV)
ISOLATED_RUNTIME = json.loads(RUNTIME_REPORT.read_text(encoding='utf-8'))
assert ISOLATED_RUNTIME['status'] == 'TF215_GPU_RUNTIME_VALIDATED'
assert ISOLATED_RUNTIME['training_performed'] is False
assert ISOLATED_RUNTIME['internal_test_loaded'] is False
assert ISOLATED_RUNTIME['plantdoc_test_loaded'] is False
print(json.dumps(ISOLATED_RUNTIME, indent=2))

## 5. Discover the five approved TRAIN/VALIDATION sources

Attach only the five private development sources used by Experiment A. Discovery is unique and fail-closed under `/kaggle/input`; TEST-like paths are forbidden.

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from training.kaggle_experiment_b import build_execution_config_b
from training.kaggle_runtime import discover_kaggle_source_roots, write_execution_config

SOURCE_ROOTS = discover_kaggle_source_roots(Path('/kaggle/input'))
print('Resolved approved source roots:')
for key, path in SOURCE_ROOTS.items():
    print(f' - {key}: {path}')

BATCH_SIZE = 32
START_TRAINING = False
INTERRUPTED_PHASE_ACTION = 'fail'
EXECUTION_CONFIG = RUNTIME_ROOT / 'experiment-b-config.json'
CONFIG = build_execution_config_b(
    SOURCE_ROOTS, batch_size=BATCH_SIZE, start_training=START_TRAINING,
    interrupted_phase_action=INTERRUPTED_PHASE_ACTION,
)
write_execution_config(EXECUTION_CONFIG, CONFIG)
print(EXECUTION_CONFIG.read_text(encoding='utf-8'))

## 6. Mandatory Experiment B preflight

Run this before considering training. It verifies the approved runtime, 58,857 TRAIN and 7,362 VALIDATION records, 39/39 coverage, baseline manifest locks, augmentation-only policy, model phases, class weights disabled, and both TEST locks.

In [ ]:
PREFLIGHT_REPORT = RUNTIME_ROOT / 'experiment-b-preflight.json'
subprocess.run([
    str(TF215_PYTHON), str(PROJECT_ROOT / 'scripts/run_kaggle_model_v2_experiment_b.py'),
    'preflight', '--config', str(EXECUTION_CONFIG), '--output', str(PREFLIGHT_REPORT),
], check=True, env=ISOLATED_ENV)
PREFLIGHT = json.loads(PREFLIGHT_REPORT.read_text(encoding='utf-8'))
DATA = PREFLIGHT['preflight']
assert PREFLIGHT['status'] == 'KAGGLE_TF215_GPU_EXPERIMENT_B_PREFLIGHT_PASSED'
assert DATA['train']['expected'] == DATA['train']['resolved'] == 58857
assert DATA['validation']['expected'] == DATA['validation']['resolved'] == 7362
assert DATA['train']['missing'] == DATA['train']['unreadable'] == 0
assert DATA['validation']['missing'] == DATA['validation']['unreadable'] == 0
assert DATA['train_class_coverage'] == DATA['validation_class_coverage'] == 39
assert DATA['augmentation_audit']['validation_augmentation_enabled'] is False
assert PREFLIGHT['class_weights'] is None
assert PREFLIGHT['training_performed'] is False
assert PREFLIGHT['internal_test_loaded'] is False
assert PREFLIGHT['plantdoc_test_loaded'] is False
print(json.dumps(PREFLIGHT, indent=2))

## 7. Human stop and dual training authorization

Stop after preflight review. The committed defaults are `False`, `False`, and `fail`. For the known Phase 1 interruption only, after reviewing epochs 1â€“5 and the checkpoint, a human may deliberately set both booleans to `True` and set `INTERRUPTED_PHASE_ACTION = 'resume'`. The runner then validates provenance, history, checkpoint identity, optimizer state, and both TEST locks before continuing. `restart` is a separate explicit action and resume never falls back to epoch 0.

In [ ]:
START_TRAINING = False
AUTHORIZE_TRAINING_CLI = False
INTERRUPTED_PHASE_ACTION = 'fail'
if not (START_TRAINING and AUTHORIZE_TRAINING_CLI):
    print('SAFE STOP: Experiment B preflight complete; training remains disabled.')
else:
    CONFIG = build_execution_config_b(
        SOURCE_ROOTS, batch_size=32, start_training=True,
        interrupted_phase_action=INTERRUPTED_PHASE_ACTION,
    )
    write_execution_config(EXECUTION_CONFIG, CONFIG)
    subprocess.run([
        str(TF215_PYTHON), str(PROJECT_ROOT / 'scripts/run_kaggle_model_v2_experiment_b.py'),
        'train', '--config', str(EXECUTION_CONFIG), '--authorize-training',
    ], check=True, env=ISOLATED_ENV)

## 8. Post-training A-vs-B comparison — VALIDATION only

After separately authorized training, `training.validation_comparison` can compare finalized A and B `validation-metrics.json` artifacts against the unchanged VALIDATION manifest. It reads no models or images, performs no inference, and rejects TEST access. Baseline values come from supplied Experiment A artifacts rather than rounded constants. The report includes overall and real-world deltas, per-class/Tomato/Potato analysis, confusion changes, and a fixed-seed paired class-aware bootstrap. Keep this optional reporting step disabled until both finalized result directories are available.

In [ ]:
RUN_VALIDATION_COMPARISON = False
if not RUN_VALIDATION_COMPARISON:
    print('VALIDATION-only A-vs-B comparison is disabled.')
else:
    EXPERIMENT_A_RESULTS = Path('/kaggle/input/REPLACE_WITH_FINALIZED_EXPERIMENT_A_RESULTS')
    EXPERIMENT_B_RESULTS = Path('/kaggle/working/agridiagnose-exp-b-results')
    COMPARISON_RESULTS = Path('/kaggle/working/agridiagnose-exp-a-vs-b-validation')
    VALIDATION_MANIFEST = PROJECT_ROOT / 'training/datasets/manifests/dataset-v2-validation.csv'
    subprocess.run([
        str(TF215_PYTHON), str(PROJECT_ROOT / 'scripts/run_kaggle_model_v2_experiment_b.py'),
        'compare-validation', '--experiment-a-dir', str(EXPERIMENT_A_RESULTS),
        '--experiment-b-dir', str(EXPERIMENT_B_RESULTS),
        '--validation-manifest', str(VALIDATION_MANIFEST),
        '--output-dir', str(COMPARISON_RESULTS),
    ], check=True, cwd=PROJECT_ROOT, env=ISOLATED_ENV)
    print('VALIDATION-only comparison artifacts:', COMPARISON_RESULTS)

## Required outcome now

Keep `START_TRAINING = False`, `AUTHORIZE_TRAINING_CLI = False`, `INTERRUPTED_PHASE_ACTION = 'fail'`, and `RUN_VALIDATION_COMPARISON = False` until a separate human decision. Preserve the existing interrupted Phase 1 checkpoint, epochs 1â€“5 history, runtime audit, and preflight report. No TEST data is needed or permitted.